# 01 · Spider data analysis and prompting baselines
**Project:** Fine-tuning a small open LLM for text-to-SQL, with execution-based evaluation.
**Base model:** `Qwen2.5-Coder-1.5B-Instruct`. **Benchmark:** Spider 1.0 (cross-database; dev DBs never appear in training).

This notebook profiles the data and measures how far **prompting alone** gets:
schema-format ablation (zero-shot) → few-shot → full-dev baselines. Runtime: **T4 GPU**, about 35–50 min.

> Spider 1.0 is downloaded automatically by the setup cell (official Google Drive file, with a Hugging Face mirror as fallback).
> Only if both fail: put the zip manually at `MyDrive/masters-projects/spider_data.zip`.

In [ ]:
# ---------- Setup: run this cell first ----------
import os, sys, subprocess, json, time, warnings
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less GPU memory fragmentation
warnings.filterwarnings("ignore")
SMOKE = os.environ.get("T2S_SMOKE") == "1"      # local CPU smoke test (ignore on Colab/Kaggle)
ON_KAGGLE = os.path.exists("/kaggle/input")
try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
from pathlib import Path
if ON_KAGGLE:
    # Code is attached as a Kaggle Dataset (read-only) -> copy it to the writable /kaggle/working
    import glob, shutil, zipfile
    PROJECT_DIR = "/kaggle/working/text2sql"

    def _walk(top="/kaggle/input"):
        return os.walk(top, followlinks=True)   # Kaggle may mount inputs via symlinks

    def _version(code_root):
        import re as _re
        m = _re.search(r"__version__\s*=\s*(\d+)", (Path(code_root) / "src/t2s/__init__.py").read_text())
        return int(m.group(1)) if m else 0

    def _find_code():
        zips, found = [], []
        for root, dirs, files in _walk():
            if root.endswith("src/t2s") and "modeling.py" in files and not root.startswith(PROJECT_DIR):
                found.append(Path(root).parents[1])
        if found:   # several attached versions -> use the newest code
            found.sort(key=_version, reverse=True)
            if len(found) > 1:
                print("several code copies found:", [(str(f), _version(f)) for f in found])
            return found[0]
        for root, dirs, files in _walk():
            zips += [Path(root) / f for f in files if f.lower().endswith(".zip") and "text2sql" in f.lower()]
        for z in zips:   # dataset kept as an un-extracted zip -> extract it ourselves
            tmp = Path("/kaggle/tmp/code"); tmp.mkdir(parents=True, exist_ok=True)
            zipfile.ZipFile(z).extractall(tmp)
            for root, dirs, files in os.walk(tmp):
                if root.endswith("src/t2s") and "modeling.py" in files:
                    return Path(root).parents[1]
        return None

    code_root = _find_code()
    if code_root is None:
        print("Could not find the text2sql code. Contents of /kaggle/input:")
        for root, dirs, files in _walk():
            depth = root.count(os.sep) - 2
            if depth <= 4:
                print("  " * depth + os.path.basename(root) + "/", (files[:5] if files else ""))
        raise RuntimeError("Code dataset not attached. Right panel -> Input -> '+ Add Input' -> Your Work -> Datasets -> "
                           "text2sql-code -> Add. Then run again. (Send the listing above if it is attached.)")
    print("code found at", code_root, "| code version", _version(code_root))
    # Kaggle images ship torchao 0.10, which makes peft refuse to load LoRA adapters on fp16 models. We don't need it.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], capture_output=True)
    shutil.copytree(code_root, PROJECT_DIR, dirs_exist_ok=True,   # always refresh the code
                    ignore=shutil.ignore_patterns("artifacts", "results", "__pycache__"))
    for m in [m for m in list(sys.modules) if m == "t2s" or m.startswith("t2s.")]:
        del sys.modules[m]   # pick up new code if the kernel was not restarted
    # Resume support: if a previous run's Output is attached as an input, reuse its results/artifacts
    for root, dirs, files in _walk():
        if os.path.basename(root) == "results" and os.path.basename(os.path.dirname(root)) == "text2sql":
            prev_root = Path(root).parent
            for sub_dir in ["results", "artifacts", "reports"]:
                if (prev_root / sub_dir).exists():
                    shutil.copytree(prev_root / sub_dir, f"{PROJECT_DIR}/{sub_dir}", dirs_exist_ok=True)
            print("restored previous results from", prev_root)
    if not SMOKE:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_DIR}/requirements-colab.txt"], check=True)
elif SMOKE:
    PROJECT_DIR = os.environ["T2S_PROJECT_DIR"]
elif not ON_COLAB:
    # Local machine (VS Code / Jupyter): the notebook lives in text2sql/notebooks/
    PROJECT_DIR = str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
else:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/masters-projects/text2sql"   # <- change if needed
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_DIR}/requirements-colab.txt"], check=True)
sys.path.insert(0, f"{PROJECT_DIR}/src")
ART, FIG, RES = f"{PROJECT_DIR}/artifacts", f"{PROJECT_DIR}/reports/figures", f"{PROJECT_DIR}/results"
for d in (ART, FIG, RES): os.makedirs(d, exist_ok=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 110
from t2s.spider import download_spider, find_spider_root, load_split, db_path, serialize_schema, schema_stats
from t2s import modeling as MD, evaluate as EV
from t2s.analysis import hardness, HARDNESS_ORDER

# ---- data ----
if SMOKE:
    from t2s.smoke import make_fake_spider
    SPIDER = find_spider_root(make_fake_spider(f"{PROJECT_DIR}/data"))
else:
    if ON_KAGGLE:
        attached = [os.path.join(r, "dev.json") for r, d, f in os.walk("/kaggle/input", followlinks=True)
                    if "dev.json" in f and os.path.isdir(os.path.join(r, "database"))]   # Spider attached as input?
        SPIDER = find_spider_root(Path(attached[0]).parent) if attached else download_spider("/kaggle/working/spider")
    elif ON_COLAB:
        SPIDER = download_spider("/content/spider", drive_zip="/content/drive/MyDrive/masters-projects/spider_data.zip")
    else:
        SPIDER = download_spider(f"{PROJECT_DIR}/data/spider", drive_zip=f"{PROJECT_DIR}/data/spider_data.zip")
train_all, dev = load_split(SPIDER, "train"), load_split(SPIDER, "dev")

# ---- model ----
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
if SMOKE:
    from t2s.smoke import make_tiny_model
    tiny = f"{ART}/tiny-model"
    if not os.path.exists(f"{tiny}/config.json"):
        make_tiny_model(tiny, [e.question + " " + e.query for e in train_all] + [serialize_schema(db_path(SPIDER, e.db_id), "ddl_rows") for e in train_all[:20] + dev])
    BASE_MODEL = tiny
DEV_SUBSET = sorted(np.random.RandomState(0).choice(len(dev), min(300, len(dev)), replace=False))  # fixed subset for ablations
SEED = 42
def savefig(name):
    plt.tight_layout(); plt.savefig(f"{FIG}/{name}.png", dpi=160, bbox_inches="tight"); plt.show()
def save_json(o, name):
    json.dump(o, open(f"{ART}/{name}.json", "w"), indent=2, default=float)
def load_json(name, default=None):
    p = f"{ART}/{name}.json"; return json.load(open(p)) if os.path.exists(p) else default
def cached(name, fn):
    """Run fn() once and store its DataFrame as results/<name>.parquet; later calls (after a crash) just load it."""
    p = f"{RES}/{name}.parquet"
    if os.path.exists(p):
        df = pd.read_parquet(p); print(f"[cached] {name}: EX={df.exec_match.mean():.4f}"); return df
    df = fn(); df.to_parquet(p, index=False); print(f"[done] {name}: EX={df.exec_match.mean():.4f}"); return df
def gen_cache(name):
    return f"{RES}/gen_{name}.jsonl"
print(f"train={len(train_all)}  dev={len(dev)}  model={BASE_MODEL}  device={MD.device_info()}")

## 1. Dataset profile

In [ ]:
dev_h = pd.Series([hardness(e) for e in dev]).value_counts().reindex(HARDNESS_ORDER)
print("train DBs:", len({e.db_id for e in train_all}), " dev DBs:", len({e.db_id for e in dev}),
      " overlap:", len({e.db_id for e in train_all} & {e.db_id for e in dev}))
st = pd.DataFrame([schema_stats(db_path(SPIDER, d)) | {"db_id": d} for d in sorted({e.db_id for e in dev})])
display(st.describe().round(1))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
dev_h.plot.bar(ax=ax[0], color="#4C72B0"); ax[0].set_title("Dev hardness (official Spider levels)")
st.n_tables.plot.hist(ax=ax[1], bins=15, color="#55A868"); ax[1].set_title("Tables per dev database")
savefig("01_data_profile")

In [ ]:
# Prompt length (tokens) under the three schema formats -> drives memory/speed and the max length for training
tok = MD.load_tokenizer(BASE_MODEL)
lens = {}
for fmt in ["compact", "ddl", "ddl_rows"]:
    chats = EV.make_chats(train_all[::10], SPIDER, fmt, with_answer=True)
    lens[fmt] = [len(tok(tok.apply_chat_template(c, tokenize=False), add_special_tokens=False)["input_ids"]) for c in chats]
L = pd.DataFrame(lens); display(L.describe(percentiles=[.5, .9, .95, .99]).round(0))
L.plot.hist(bins=40, alpha=.6, figsize=(7, 3.5)); plt.xlabel("tokens per training example"); savefig("01_prompt_lengths")

## 2. Zero-shot: which schema format works best?
Same model and question; only the schema format changes (fixed 300-question dev subset):
* `compact`: `table(col1, col2*)` plus foreign keys
* `ddl`: `CREATE TABLE` statements with types, primary keys and foreign keys
* `ddl_rows`: DDL plus 3 example rows per table (Rajkumar et al., 2022)

In [ ]:
model = MD.load_model(BASE_MODEL, load_4bit=False)     # 1.5B in fp16 = ~3 GB; faster than 4-bit for inference
sub = [dev[i] for i in DEV_SUBSET]
fmt_results = {}
for fmt in ["compact", "ddl", "ddl_rows"]:
    fmt_results[f"zero-shot [{fmt}]"] = cached(f"subset_zero_{fmt}", lambda: EV.score(
        sub, MD.generate(model, tok, EV.make_chats(sub, SPIDER, fmt), batch_size=16, cache_path=gen_cache(f"subset_zero_{fmt}")), SPIDER))
fmt_table = EV.compare_table(fmt_results); fmt_table.round(3)

In [ ]:
BEST_FMT = fmt_table.exec_acc.idxmax().split("[")[1].rstrip("]")
save_json({"best_format": BEST_FMT, "table": fmt_table.reset_index().to_dict(orient="records")}, "schema_format_ablation")
print("best schema format:", BEST_FMT)

## 3. Few-shot (3 demonstrations from training databases)

In [ ]:
shots = MD.pick_fewshot(train_all, k=3, seed=SEED)
for s in shots: print(f"[{s.db_id}] {s.question}\n   {s.query}")
fmt_results["3-shot"] = cached("subset_3shot", lambda: EV.score(sub, MD.generate(
    model, tok, EV.make_chats(sub, SPIDER, BEST_FMT, fewshot=shots), batch_size=8, cache_path=gen_cache("subset_3shot")), SPIDER))
EV.compare_table(fmt_results).round(3)

## 4. Full-dev baselines (1,034 questions)
Zero-shot on the full dev set. 3-shot on the full dev set is optional (`FULL_DEV_3SHOT`): its long prompts make it the
slowest step (~75 min on a T4), and the 300-question subset already measures it.

In [ ]:
FULL_DEV_3SHOT = False
full = {"zero-shot": cached("dev_zero-shot", lambda: EV.score(dev, MD.generate(
    model, tok, EV.make_chats(dev, SPIDER, BEST_FMT), batch_size=16, cache_path=gen_cache("dev_zero-shot")), SPIDER))}
if FULL_DEV_3SHOT:
    full["3-shot"] = cached("dev_3-shot", lambda: EV.score(dev, MD.generate(
        model, tok, EV.make_chats(dev, SPIDER, BEST_FMT, fewshot=shots), batch_size=8, cache_path=gen_cache("dev_3-shot")), SPIDER))
save_json({"fewshot_examples": [vars(s) | {"sql": None} for s in shots]}, "fewshot")
EV.compare_table(full).round(4)

## 5. (Optional) Reference: a 5× larger model, zero-shot
`Qwen2.5-Coder-7B-Instruct` in 4-bit on the 300-question subset. It answers: *can fine-tuning a 1.5B model match a 7B one?*

In [ ]:
RUN_7B = not SMOKE        # set False to skip (~20-30 min on a T4)
if RUN_7B and not os.path.exists(f"{RES}/subset_7b_zero_shot.parquet"):
    import gc, torch
    del model; gc.collect(); torch.cuda.empty_cache()
    big = "Qwen/Qwen2.5-Coder-7B-Instruct"
    m7, tok7 = MD.load_model(big, load_4bit=True), MD.load_tokenizer(big)
    r7 = cached("subset_7b_zero_shot", lambda: EV.score(sub, MD.generate(
        m7, tok7, EV.make_chats(sub, SPIDER, BEST_FMT), batch_size=8, cache_path=gen_cache("subset_7b")), SPIDER))
    del m7; gc.collect(); torch.cuda.empty_cache()

In [ ]:
pd.concat({k: v for k, v in fmt_results.items()}).reset_index(level=0).rename(columns={"level_0": "system"}) \
  .to_parquet(f"{RES}/subset_prompting.parquet", index=False)
print("saved.")